# Relatório FOMC – Mesa de Investimentos / Depin

Notebook para geração automática do relatório Word do FOMC.

**Fluxo:**
1. Configurar data da reunião e a pasta do dia
2. Download automático de documentos do Fed (se necessário)
3. Parse do statement (decisão, taxa, votação)
4. Projeções SEP (se reunião trimestral: Mar/Jun/Set/Dez)
5. Insumos do dia
6. Resumo da decisão
7. Resumo pós-coletiva
8. Bancos
9. Revisão
10. Preview
11. Word

In [ ]:
# Setup + Imports
from pathlib import Path

from reports._paths import INPUT, OUTPUT

import warnings
warnings.filterwarnings('ignore')

from datetime import datetime

import pandas as pd
from IPython.display import display, Markdown

from reports.fomc.core.data_loader import (
    get_fomc_dates,
    get_meeting_documents,
    is_sep_meeting,
    get_prior_sep_projections,
    load_dots_history,
    load_dots_snapshot,
    save_sep_to_excel,
    fetch_market_reaction,
    _TZ_ET,
    _TZ_BRT,
)
from reports.fomc.core.pdf_parser import (
    parse_projection_table,
    parse_statement,
    get_projection_dates,
)
from reports.fomc.core.calculations import (
    get_prior_sep_meeting_date,
    get_prior_meeting_label,
)
from reports.fomc.core.fed_scraper import (
    fetch_latest_documents,
    check_for_updates,
)
from reports.fomc.core.word_export import create_market_reaction_grid
from reports.fomc.core.word_report import generate_fomc_report

from reports.fomc.core.drafting import (
    MeetingInputs,
    DraftingError,
    draft_summary,
    draft_bank_comments,
    review_report,
    read_headlines,
    read_presser,
    read_presser_transcript,
    read_bank_sources,
    headlines_for_report,
    market_snapshot,
)

print("Módulos carregados com sucesso.")

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO USUÁRIO
# ============================================================

# Data da reunião (formato YYYYMMDD)
MEETING_DATE = '20261028'

# Pasta do dia: input/informes_eventos/fomc/<data>/ com headlines.txt, coletiva.txt e bancos/*.pdf.
# Resumo, headlines e comentários dos bancos NÃO ficam mais neste notebook: vêm
# dos arquivos que você cola ali e das etapas de modelo, nas células mais abaixo.

# Qual resumo vai ao Word: "auto" pega o mais avançado que existir na pasta de
# saída (revisão > coletiva > decisão); ou force "revisao", "coletiva", "decisao".
SUMMARY_SOURCE = "auto"

# --- Reação de mercado (grid intraday 3x3 GERADO em Python) ---
# O horário da decisão é FIXO em horário de NY (ET); a conversão para Brasília
# é automática (auto-DST via zoneinfo). Edite apenas se o Fed mudar o horário.
FOMC_DECISION_TIME_ET = "14:00:00"

# Eixo X do grid se estende até este horário (em Brasília), mesmo sem dados até lá.
GRID_X_END_BRT = "20:00:00"

# Fallback: screenshot manual, usado SÓ se o Bloomberg estiver indisponível.
MARKET_REACTION_PNG = r"c:\Users\mmart\OneDrive\BCB\dirin\15_informes\fomc\screenshots\market_reaction.png"

# Dot plot screenshot (só SEP, ou None para reuniões sem SEP)
DOT_PLOT_PNG = r"c:\Users\mmart\OneDrive\BCB\dirin\15_informes\fomc\screenshots\dot_plot.png"

# ============================================================
from reports.fomc.core import drafting
DAY_FOLDER = drafting.day_folder(MEETING_DATE)
OUT_FOLDER = drafting.output_folder(MEETING_DATE)
print(f"Reunião configurada: {MEETING_DATE}")
print(f"Tipo: {'COM SEP' if is_sep_meeting(MEETING_DATE) else 'SEM SEP'}")
print(f"Pasta do dia: {DAY_FOLDER} ({'existe' if DAY_FOLDER.is_dir() else 'NÃO EXISTE — crie e cole os headlines'})")

In [ ]:
# Download automático de documentos do Fed
print("Verificando documentos disponíveis...\n")

updates = check_for_updates()
for doc_type, has_update in updates.items():
    status = "NOVO disponível" if has_update else "Atualizado"
    print(f"  {doc_type}: {status}")

# Download de documentos faltantes
if any(updates.values()):
    print("\nBaixando documentos novos...")
    downloaded = fetch_latest_documents()
    for doc_type, path in downloaded.items():
        if path:
            print(f"  ✓ {doc_type}: {path.name}")
else:
    print("\nTodos os documentos estão atualizados.")

# Listar documentos da reunião
documents = get_meeting_documents(MEETING_DATE)
print(f"\nDocumentos para reunião {MEETING_DATE}:")
for doc_type, path in documents.items():
    status = f"✓ {path.name}" if path else "✗ Não encontrado"
    print(f"  {doc_type}: {status}")

In [ ]:
# Parse do statement
statement_data = {}
if documents['statement']:
    statement_data = parse_statement(documents['statement'])

    print("=" * 50)
    print("DECISÃO DO FOMC")
    print("=" * 50)
    decision_map = {"hold": "MANUTENÇÃO", "cut": "CORTE", "hike": "ALTA"}
    print(f"Decisão: {decision_map.get(statement_data.get('decision', ''), 'N/A')}")
    print(f"Taxa alvo: {statement_data.get('target_rate_low', 'N/A')}% - {statement_data.get('target_rate_high', 'N/A')}%")
    print(f"Unânime: {'Sim' if statement_data.get('unanimous') else 'Não'}")
    if statement_data.get('dissenters'):
        print(f"Dissidentes: {', '.join(statement_data['dissenters'])}")
    if statement_data.get('key_phrases'):
        print(f"\nFrases-chave:")
        for phrase in statement_data['key_phrases']:
            print(f"  - {phrase}")
    print("=" * 50)
else:
    print("Statement não disponível para esta reunião.")

In [ ]:
# ============================================================
# REAÇÃO DE MERCADO — grid intraday 3x3 (gerado em Python)
# ============================================================
# Converte o horário da decisão (ET fixo) para Brasília na data da reunião,
# busca os dados intraday no Bloomberg (dia inteiro, desde a abertura da Ásia) e
# gera o grid, salvando o PNG que será inserido no relatório. Se o Bloomberg
# estiver indisponível, mantém o screenshot manual configurado em MARKET_REACTION_PNG.
import matplotlib.pyplot as plt

def _et_to_brt(meeting_date: str, hms: str) -> pd.Timestamp:
    """Converte 'HH:MM:SS' em ET (na data da reunião) para Timestamp em Brasília."""
    md = datetime.strptime(meeting_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(md, t), tz=_TZ_ET).tz_convert(_TZ_BRT)

def _brt_time(meeting_date: str, hms: str) -> pd.Timestamp:
    """Timestamp em Brasília na data da reunião a partir de 'HH:MM:SS' (já em BRT)."""
    md = datetime.strptime(meeting_date, "%Y%m%d").date()
    t = datetime.strptime(hms, "%H:%M:%S").time()
    return pd.Timestamp(datetime.combine(md, t), tz=_TZ_BRT)

# Apenas a linha da decisão (as linhas da coletiva foram removidas)
event_times = {"decisao": _et_to_brt(MEETING_DATE, FOMC_DECISION_TIME_ET)}
axis_end = _brt_time(MEETING_DATE, GRID_X_END_BRT)

print(f"Decisão FOMC (horário de Brasília): {event_times['decisao']:%H:%M}")
print(f"Eixo X até: {axis_end:%H:%M} BRT")

grid_png = OUTPUT / 'reports' / 'fomc' / f'market_reaction_{MEETING_DATE}.png'
market_data = None
try:
    market_data = fetch_market_reaction(MEETING_DATE)
    fig = create_market_reaction_grid(
        market_data,
        event_times=event_times,
        meeting_date=MEETING_DATE,
        output_path=grid_png,
        axis_end=axis_end,
    )
    MARKET_REACTION_PNG = str(grid_png)
    print(f"\n✓ Grid de reação de mercado gerado: {grid_png}")
    display(fig)
    plt.close(fig)  # evita exibição duplicada pelo backend inline do Jupyter
except RuntimeError as e:
    print(f"\n⚠️ Bloomberg indisponível ({e}).")
    print(f"   Usando screenshot manual: {MARKET_REACTION_PNG}")

In [ ]:
# Projeções SEP (condicional — só para reuniões Mar/Jun/Set/Dez)
sep_current = None
sep_prior = None
prior_label = None
current_ct = None
prior_ct = None

if is_sep_meeting(MEETING_DATE):
    print("Reunião COM SEP — processando projeções...\n")

    # Parse completo do PDF de projeções
    if documents.get('projections'):
        proj_data = parse_projection_table(documents['projections'])
        sep_current = proj_data.get('medians')
        sep_prior = proj_data.get('prior_medians')
        prior_label = proj_data.get('prior_label', 'Prior projection')
        current_ct = proj_data.get('central_tendency')
        prior_ct = proj_data.get('prior_ct')

        if sep_current is not None and not sep_current.empty:
            print("Projeções atuais (mediana):")
            display(sep_current)

        if sep_prior is not None and not sep_prior.empty:
            print(f"\nProjeções anteriores ({prior_label}):")
            display(sep_prior)

        if current_ct is not None and not current_ct.empty:
            print("\nTendência Central (atual):")
            display(current_ct)
    else:
        # Fallback: carregar projeções de fontes alternativas
        prior_date = get_prior_sep_meeting_date(MEETING_DATE)
        if prior_date:
            prior_label = get_prior_meeting_label(prior_date)
            print(f"Carregando projeções anteriores ({prior_label}, {prior_date})...")
            sep_prior_fallback = get_prior_sep_projections(MEETING_DATE)
            if sep_prior_fallback is not None and not sep_prior_fallback.empty:
                sep_prior = sep_prior_fallback
                print("Projeções anteriores carregadas (fallback):")
                display(sep_prior)

    print(f"\nAnos disponíveis: {proj_data.get('year_columns', [])}" if documents.get('projections') else "")
else:
    print("Reunião SEM SEP — seção de projeções será omitida no relatório.")

In [ ]:
# Evolução histórica dos Dots (Bloomberg) + Auto-save SEP.xlsx
if is_sep_meeting(MEETING_DATE):
    # --- DOTS Bloomberg ---
    dots_hist = None
    try:
        dots_hist = load_dots_history()
        print("Evolução histórica dos DOTS Medians (Bloomberg):")
        display(dots_hist.tail(10))
    except RuntimeError as e:
        print(f"Bloomberg não disponível para DOTS: {e}")
        try:
            dots_snap = load_dots_snapshot()
            if dots_snap:
                print("DOTS Snapshot (grid1.xlsx):")
                for ticker, val in dots_snap.items():
                    print(f"  {ticker}: {val}")
            else:
                print("Nenhum dado de DOTS disponível.")
        except RuntimeError:
            print("grid1.xlsx não encontrado.")

    # --- Auto-save SEP.xlsx ---
    if sep_current is not None and not sep_current.empty:
        try:
            sep_path = save_sep_to_excel(
                MEETING_DATE, sep_current, sep_prior, prior_label or "Prior projection",
                ct=current_ct, ranges=proj_data.get('range') if documents.get('projections') else None,
            )
            print(f"\nSEP salvo em: {sep_path}")
        except Exception as e:
            print(f"Erro ao salvar SEP.xlsx: {e}")

## Etapas de modelo

Cada célula abaixo chama o modelo uma vez e grava a resposta inteira em
`output/informes_eventos/reports/fomc/<data>/`. Nada roda sozinho: execute a célula quando o insumo
dela existir. Reexecutar sobrescreve o `.md` da etapa.

In [ ]:
# Insumos do dia: o que as células anteriores produziram + a pasta do dia.
# Não chama o modelo. Diz o que achou e o que falta.
_statement_text = None
if documents.get('statement'):
    import pdfplumber
    with pdfplumber.open(documents['statement']) as _pdf:
        _statement_text = "\n".join((p.extract_text() or "") for p in _pdf.pages)

_market = None
if market_data is not None and not market_data.empty:
    _market = market_snapshot(market_data, event_times["decisao"])

# headlines.txt é opcional: o autor cola os headlines direto no Word.
_headlines = read_headlines(DAY_FOLDER / "headlines.txt") if (DAY_FOLDER / "headlines.txt").is_file() else []

# Research dos bancos: bancos.txt (blocos "Casa abaixo:") e bancos/*.pdf|*.txt.
# Alimenta o texto principal, atribuído à casa. Opcional.
BANK_SOURCES, _ignored = read_bank_sources(DAY_FOLDER)
for _name in _ignored:
    print(f"⚠️ research ignorado: {_name}")

_presser = read_presser(DAY_FOLDER)
_transcript = read_presser_transcript(documents.get("presser"))

INPUTS = MeetingInputs(
    meeting_date=MEETING_DATE,
    statement_text=_statement_text,
    statement=statement_data or None,
    is_sep=is_sep_meeting(MEETING_DATE),
    sep_medians=sep_current,
    sep_prior=sep_prior,
    prior_label=prior_label,
    headlines=_headlines,
    market=_market,
)

print(f"Statement: {'✓' if _statement_text else '✗'}")
print(f"SEP: {'✓' if INPUTS.is_sep and sep_current is not None else '— (sem SEP)' if not INPUTS.is_sep else '✗'}")
print(f"Headlines: {len(_headlines)} ({sum(h.bold for h in _headlines)} em destaque)" if _headlines else "Headlines: — (opcional; cole no Word)")
print(f"Coletiva (headlines): {len(_presser) if _presser else '✗ (coletiva.txt ausente)'}")
print(f"Coletiva (transcrição do Fed): {'✓ ' + documents['presser'].name if _transcript else '✗ (FOMCpresconf ausente; a célula 3 baixa quando o Fed publicar)'}")
print(f"Reação de mercado: {'✓ até ' + _market.last_time if _market else '✗ (sem Bloomberg)'}")
print(f"Research: {', '.join(b.name for b in BANK_SOURCES) if BANK_SOURCES else '— (opcional; bancos.txt ou bancos/)'}")

In [ ]:
# Etapa: resumo da decisão. Requer o statement; headlines e research são opcionais.
summary_decision = draft_summary(INPUTS, stage="decision", bank_sources=BANK_SOURCES)
display(Markdown(summary_decision))
print(f"\nGravado em: {OUT_FOLDER / 'resumo_decisao.md'}")

In [ ]:
# Etapa: resumo pós-coletiva. Requer o resumo da decisão gravado e ao menos uma
# fonte da coletiva: a transcrição do Fed (documents['presser'], baixada pela
# célula 3) ou os headlines colados em coletiva.txt. Só os headlines vão ao Word.
_presser = read_presser(DAY_FOLDER)
_transcript = read_presser_transcript(documents.get("presser"))
BANK_SOURCES, _ignored = read_bank_sources(DAY_FOLDER)  # relido: pode ter blocos novos
if _presser is None and _transcript is None:
    print("Nem coletiva.txt nem FOMCpresconf<data>.pdf existem ainda — pule esta célula por enquanto.")
else:
    _prev_path = OUT_FOLDER / "resumo_decisao.md"
    if not _prev_path.is_file():
        raise DraftingError(f"Run the decision summary first: {_prev_path} not found")
    from reports.fomc.core.drafting import extract_fenced_block
    _previous = extract_fenced_block(_prev_path.read_text(encoding="utf-8"))
    # A reação de mercado é refeita aqui para pegar o que veio depois da coletiva,
    # e o grid é regerado com os mesmos argumentos da célula de reação de mercado.
    if market_data is not None and not market_data.empty:
        try:
            market_data = fetch_market_reaction(MEETING_DATE)
            INPUTS.market = market_snapshot(market_data, event_times["decisao"])
            _fig = create_market_reaction_grid(
                market_data,
                event_times=event_times,
                meeting_date=MEETING_DATE,
                output_path=grid_png,
                axis_end=axis_end,
            )
            plt.close(_fig)
            print(f"✓ Grid de reação de mercado regerado: {grid_png}")
        except RuntimeError as e:
            print(f"⚠️ Bloomberg indisponível ({e}); usa a reação já carregada.")
    summary_presser = draft_summary(
        INPUTS, stage="presser", previous=_previous,
        presser_headlines=_presser, presser_transcript=_transcript,
        bank_sources=BANK_SOURCES,
    )
    display(Markdown(summary_presser))
    print(f"\nGravado em: {OUT_FOLDER / 'resumo_coletiva.md'}")

In [ ]:
# Etapa opcional: seção 'Comentários dos bancos' no Word, um parágrafo por casa.
# O research já entra no texto principal; rode esta célula só se quiser a seção.
if not BANK_SOURCES:
    print("Sem research em bancos.txt ou bancos/ — pule esta célula.")
    bank_comments = {}
else:
    bank_comments = draft_bank_comments(INPUTS, BANK_SOURCES)
    for _bank, _text in bank_comments.items():
        display(Markdown(f"**{_bank}**: {_text}"))
    print(f"\nGravado em: {OUT_FOLDER / 'bancos.md'}")

In [ ]:
# Etapa: revisão de coerência sobre o resumo escolhido por SUMMARY_SOURCE.
# review_source nunca deixa a revisão ler a si mesma: "auto" e "revisao" caem
# para coletiva-senão-decisão; "coletiva"/"decisao" explícitos valem direto.
from reports.fomc.core.drafting import (
    extract_fenced_block,
    parse_bank_sections,
    pick_summary,
    review_source,
)

_text, _from = pick_summary(OUT_FOLDER, review_source(OUT_FOLDER, SUMMARY_SOURCE))
_banks_path = OUT_FOLDER / "bancos.md"
_banks = parse_bank_sections(extract_fenced_block(_banks_path.read_text(encoding="utf-8"))) if _banks_path.is_file() else None
review_report(
    INPUTS, _text, _banks,
    presser_headlines=read_presser(DAY_FOLDER),
    presser_transcript=read_presser_transcript(documents.get("presser")),
    bank_sources=BANK_SOURCES,
)
print(f"Revisado a partir de: {_from.name}")
display(Markdown((OUT_FOLDER / "revisao.md").read_text(encoding="utf-8")))

In [ ]:
# Preview do relatório: de onde vem cada texto.
from reports.fomc.core.drafting import extract_fenced_block, parse_bank_sections, pick_summary

SUMMARY_TEXT_FINAL, SUMMARY_FROM = pick_summary(OUT_FOLDER, SUMMARY_SOURCE)
_hl_path = DAY_FOLDER / "headlines.txt"
HEADLINES_FINAL = headlines_for_report(read_headlines(_hl_path)) if _hl_path.is_file() else []
_presser_final = read_presser(DAY_FOLDER)
PRESSER_FINAL = headlines_for_report(_presser_final) if _presser_final else None
_banks_path = OUT_FOLDER / "bancos.md"
BANKS_FINAL = parse_bank_sections(extract_fenced_block(_banks_path.read_text(encoding="utf-8"))) if _banks_path.is_file() else {}

is_sep = is_sep_meeting(MEETING_DATE)
dt = pd.to_datetime(MEETING_DATE, format='%Y%m%d')
print("=" * 60)
print("PREVIEW DO RELATÓRIO")
print("=" * 60)
print(f"\n📄 Mesa de Investimentos / Depin — Reunião do FOMC – {dt.strftime('%d/%m/%Y')}")
print(f"   Tipo: {'COM SEP' if is_sep else 'SEM SEP'}")
print(f"\n--- CAPA ---")
print(f"  Resumo: {len(SUMMARY_TEXT_FINAL.split())} palavras, de {SUMMARY_FROM.name}")
print(f"  Headlines: {len(HEADLINES_FINAL)} itens ({sum(1 for h in HEADLINES_FINAL if h['bold'])} em destaque)" if HEADLINES_FINAL else "  Headlines: — (cole no Word)")
if is_sep:
    print(f"\n--- SEÇÕES SEP ---")
    print(f"  Dot plot: {'✓' if DOT_PLOT_PNG and Path(DOT_PLOT_PNG).exists() else '✗ (screenshot necessário)'}")
    print(f"  Tabela SEP: {'✓ Atual + Anterior' if sep_prior is not None else '✓ Apenas atual' if sep_current is not None else '✗'}")
    print(f"  Coletiva: {len(PRESSER_FINAL) if PRESSER_FINAL else '— (sem coletiva.txt)'}")
print(f"\n--- REAÇÃO DE MERCADO ---")
print(f"  Grid/Imagem: {'✓' if MARKET_REACTION_PNG and Path(MARKET_REACTION_PNG).exists() else '✗ (gerar grid ou apontar screenshot)'}")
print(f"\n--- COMENTÁRIOS DOS BANCOS ---")
for bank in BANKS_FINAL or ["(nenhum)"]:
    print(f"  • {bank}")
print("\n" + "=" * 60)

In [ ]:
# GERAR RELATÓRIO WORD — a partir do que o preview escolheu.
output_dir = OUTPUT / 'reports' / 'fomc'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"Mesa de Investimentos - FOMC_{MEETING_DATE}.docx"

result_path = generate_fomc_report(
    meeting_date=MEETING_DATE,
    summary_text=SUMMARY_TEXT_FINAL,
    headlines=HEADLINES_FINAL,
    market_reaction_path=MARKET_REACTION_PNG,
    bank_comments=BANKS_FINAL,
    dot_plot_path=DOT_PLOT_PNG if is_sep_meeting(MEETING_DATE) else None,
    sep_current=sep_current,
    sep_prior=sep_prior,
    prior_meeting_label=prior_label,
    presser_headlines=PRESSER_FINAL if is_sep_meeting(MEETING_DATE) else None,
    current_ct=current_ct,
    prior_ct=prior_ct,
    include_ct=True,
    output_path=output_path,
)
print(f"Relatório gerado com sucesso!")
print(f"Arquivo: {result_path}")